# STEP 05 — External cohort provenance, metadata and cross-dataset overlap audit

**Reviewer 1, Comment 4; Reviewer 2, Comment 1.** Read-only audit; **no GPU or model inference or training**. It inspects the same frozen 100 primary and 66 MOPG-7 radiographs, verifies the reference-label counts, records image dimensions and metadata availability, and checks exact and perceptual image duplicates. The selection eligibility table is a **template**: unknown age, clinic, patient and eligibility information will stay blank/unknown rather than fabricated.

**IMPORTANT:** A negative hash/perceptual-match screen cannot establish patient-level independence or the 66-image source inclusion/exclusion criteria. Those need a genuine selection log and source metadata. Do not claim the audit establishes that the external cohort is pediatric or multicenter by itself.

Expected project root: `/content/drive/MyDrive/Pediatric_Caries_AI`. Output ZIP under `reviewer_revision/Step_05_...`.

In [ ]:
%pip -q install ImageHash

from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageOps
import imagehash, hashlib, json, zipfile, collections, re, traceback, os
import pandas as pd
import numpy as np

ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
assert ROOT.is_dir(), f'Project root not found: {ROOT}'
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=ROOT/'reviewer_revision'/f'Step_05_Cohort_Provenance_{RUN_ID}'
OUT.mkdir(parents=True, exist_ok=False)
SUFFIXES={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
print('Saving results to:', OUT)


def image_paths(folder):
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUFFIXES) if folder.is_dir() else []

PRIMARY=ROOT/'processed_data/yolo_caries_baseline'
EXTERNAL=ROOT/'external_validation/MOPG7_66'
train_items=image_paths(PRIMARY/'images/train')
train_labels=PRIMARY/'labels/train'
source_of_train='original processed_data/images/train'
if len(train_items)!=70:
    fold_val=[]
    for k in range(1,6):
        fold_val.extend(image_paths(ROOT/f'cross_validation/fold_{k}/images/val'))
    by_name={p.name:p for p in fold_val}
    if len(by_name)==70:
        train_items=sorted(by_name.values(), key=lambda p:p.name)
        train_labels=None
        source_of_train='union of cross_validation/fold_1...5/images/val'
    else:
        raise RuntimeError(f'Cannot locate complete 70-image development set; train dir has {len(train_items)} and validation union has {len(by_name)}. Do not generate a partial cohort audit.')

cohorts={
    'development':train_items,
    'internal_test':image_paths(PRIMARY/'images/test'),
    'external':image_paths(EXTERNAL/'images/test')
}
assert {k:len(v) for k,v in cohorts.items()} == {'development':70,'internal_test':30,'external':66}, {k:len(v) for k,v in cohorts.items()}
print('PASS cohort image counts:', {k:len(v) for k,v in cohorts.items()})
print('Development input source:',source_of_train)


In [ ]:
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def label_path(cohort,p):
    if cohort=='internal_test':return PRIMARY/'labels/test'/f'{p.stem}.txt'
    if cohort=='external':return EXTERNAL/'labels/test'/f'{p.stem}.txt'
    if train_labels is not None:return train_labels/f'{p.stem}.txt'
    parent=p.parent.parent.parent
    return parent/'labels/val'/f'{p.stem}.txt'

rows=[]; issues=[]
for cohort,items in cohorts.items():
    for ip in items:
        lp=label_path(cohort,ip)
        if not lp.exists():
            issues.append({'cohort':cohort,'image':ip.name,'problem':'missing label file; empty label may be expected but must be documented'})
            label_lines=[]
        else:label_lines=[s.strip() for s in lp.read_text().splitlines() if s.strip()]
        gt=[]
        for j,line in enumerate(label_lines):
            spl=line.split()
            if len(spl)!=5:
                issues.append({'cohort':cohort,'image':ip.name,'problem':f'bad label field count, line {j+1}'});continue
            try:
                c=int(float(spl[0]));cx,cy,w,h=map(float,spl[1:])
                if c!=0 or w<=0 or h<=0 or min(cx,cy,w,h)<0 or max(cx,cy,w,h)>1:raise ValueError('unexpected class or normalized geometry')
                gt.append((cx,cy,w,h))
            except Exception as exc:
                issues.append({'cohort':cohort,'image':ip.name,'problem':f'bad label line {j+1}: {exc}'})
        try:
            with Image.open(ip) as img:
                width,height=img.size
                mode=img.mode
                exif=img.getexif()
                canonical=ImageOps.exif_transpose(img).convert('L')
                p_hash=str(imagehash.phash(canonical)); d_hash=str(imagehash.dhash(canonical))
                exif_present=bool(exif)
        except Exception as exc:
            issues.append({'cohort':cohort,'image':ip.name,'problem':f'image decode failure: {exc}'});raise
        rows.append({'cohort':cohort,'image':ip.name,'width':width,'height':height,'aspect_ratio':width/height,
                     'mode':mode,'exif_present':exif_present,'source_label_count':len(gt),
                     'source_label_negative':len(gt)==0,
                     'median_box_normalized_area':float(np.median([w*h for _,_,w,h in gt])) if gt else np.nan,
                     'image_sha256':sha256_file(ip),'label_sha256':sha256_file(lp) if lp.exists() else '',
                     'pHash':p_hash,'dHash':d_hash,'source_image_path_relative_to_project':str(ip.relative_to(ROOT))})
meta=pd.DataFrame(rows)
meta.to_csv(OUT/'all_image_metadata_manifest.csv',index=False)
pd.DataFrame(issues,columns=['cohort','image','problem']).to_csv(OUT/'input_quality_issues.csv',index=False)
expected_counts={'development':206,'internal_test':108,'external':55}
counts=meta.groupby('cohort')['source_label_count'].sum().to_dict()
assert counts==expected_counts, f'Source labels changed? Expected {expected_counts}; observed {counts}; audit stopped.'
print('PASS original source box totals:',counts)

# Match ALL 100 primary images to ALL 66 external images.
primary=meta[meta.cohort!='external'].reset_index(drop=True)
ext=meta[meta.cohort=='external'].reset_index(drop=True)
match=[]
for _,p in primary.iterrows():
    for _,e in ext.iterrows():
        equal_sha=p.image_sha256==e.image_sha256
        ph=(int(p.pHash,16)^int(e.pHash,16)).bit_count()
        dh=(int(p.dHash,16)^int(e.dHash,16)).bit_count()
        if equal_sha or ph<=8 or dh<=6:
            match.append({'primary_cohort':p.cohort,'primary_image':p.image,'external_image':e.image,
                          'exact_sha256_equal':bool(equal_sha),'phash_hamming':ph,'dhash_hamming':dh,
                          'review_needed':True,'reason':'candidate visual similarity; NOT proof of patient identity'})
match_df=pd.DataFrame(match,columns=['primary_cohort','primary_image','external_image','exact_sha256_equal','phash_hamming','dhash_hamming','review_needed','reason'])
match_df.to_csv(OUT/'cross_dataset_duplicate_candidates.csv',index=False)
print('Screened 100 × 66 = 6600 cross-dataset image pairs.')
print('Exact duplicates:',sum(x['exact_sha256_equal'] for x in match))
print('Near-match candidate pairs:',len(match_df))

# Explicitly retain blank/unknown source info; there is no retrospective invention of selection criteria.
elig=meta[meta.cohort=='external'][['image','width','height','mode','source_label_count','source_label_negative']].copy()
elig['source_dataset']='MOPG-7 (selected 66-image study subset)'
for c in ['source_subject_or_patient_id','source_age','source_dentition','source_clinic','source_acquisition_device',
          'documented_inclusion_criterion','documented_exclusion_criterion','original_source_selection_log','expert_eligibility_check']:
    elig[c]='NOT DOCUMENTED IN THIS AUDIT'
elig['notes']='Image properties measured; source clinical/acquisition metadata must be separately verified.'
elig.to_csv(OUT/'external_eligibility_and_metadata_GAPS.csv',index=False)

source_roots=[ROOT/'external_validation',EXTERNAL,PRIMARY]
files=[]
for directory in source_roots:
    if not directory.is_dir():continue
    for f in directory.glob('*'):
        if f.is_file() and f.suffix.lower() in ('.csv','.xlsx','.xls','.json','.yaml','.yml','.md','.txt','.pdf'):
            files.append({'directory_relative_to_project':str(directory.relative_to(ROOT)),'filename':f.name,'size_bytes':f.stat().st_size,
                          'note':'Potential supporting metadata; file content not assessed'})
pd.DataFrame(files,columns=['directory_relative_to_project','filename','size_bytes','note']).to_csv(OUT/'source_sidecar_file_inventory.csv',index=False)

summary=meta.groupby('cohort',sort=False).agg(n_images=('image','size'),source_positives=('source_label_negative',lambda x:int((~x).sum())),source_negatives=('source_label_negative','sum'),n_boxes=('source_label_count','sum'),width_min=('width','min'),width_median=('width','median'),width_max=('width','max'),height_min=('height','min'),height_median=('height','median'),height_max=('height','max'),ratio_median=('aspect_ratio','median')).reset_index()
summary.to_csv(OUT/'cohort_summary.csv',index=False)
print(summary.to_string(index=False))

missing={'age':'Not derivable from JPEG/PNG pixels','dentition':'Requires independent expert review records',
         'patient_overlap':'Cannot be ruled out from image hashes; de-identified patient IDs required',
         'selection_eligibility':'Requires original selection and exclusion log; not recoverable from selected 66 alone',
         'acquisition_site_device':'Requires original source documentation; image dimensions are not device identity'}
(OUT/'clinical_metadata_limitations.json').write_text(json.dumps(missing,indent=2))
(OUT/'README_INTERPRETATION.txt').write_text(
"""STEP 05 — COHORT PROVENANCE & CROSS-DATASET IMAGE DUPLICATION SCREEN (WORKING)
Image hashes and pHash/dHash of 100 primary + 66 external images have been screened.
An exact hash match proves identical file bytes; pHash/dHash similarities are only candidates and may match distinct panoramic views.
Negative image-similarity results do NOT establish that patients do not overlap.
No unknown age, dentition stage, clinic, patient identity, acquisition device, or cohort-selection criteria have been guessed.
The output eligibility CSV is a gap log/template; do not cite its missing fields as clinically verified negatives.
All source annotation files were read without modification. This does not correct expert-identified source-label mismatches.
""")
archive=OUT/'Step_05_Cohort_Provenance_Results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file() and f!=archive:z.write(f,f.name)
print('\nSUCCESS. ZIP:',archive,'\nBytes:',archive.stat().st_size)
